### Étude et exploration des données brutes

Importation des librairies

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

Chargement des données

In [ ]:
df = pd.read_csv("../data/data.csv")

In [ ]:
df.head()

Dimensions

In [ ]:
df.shape

types

In [ ]:
df.info()

Statistiques descriptives

In [ ]:
df.describe()

Valeurs manquantes

In [ ]:
df.isnull().sum()

In [ ]:
(df.isnull().mean() * 100).round(2)

In [ ]:
missing = pd.DataFrame({
    "Nombre": df.isnull().sum(),
    "Pourcentage": (df.isnull().mean() * 100).round(2)
})

missing[missing["Nombre"] > 0]

Duplicates

In [ ]:
df.duplicated().sum()

Valeurs invalides

In [ ]:
#nbr des rows have quantity <0
(df["Quantity"] < 0).sum()

In [ ]:
# nbr des rows have quantity =0
(df["Quantity"] == 0).sum()

In [ ]:
#unitPrice négatif
(df["UnitPrice"] < 0).sum()

In [ ]:
# count unitPrice = 0
(df["UnitPrice"] == 0).sum()

Analyse des dates

In [ ]:
df["InvoiceDate"].dtype
#type de date StringDtype

In [ ]:
#see pattern de date 
df["InvoiceDate"].head()

Analyse des variables catégorielles

In [ ]:
df["Country"].nunique()

In [ ]:
#country existe and how much repéte 
df["Country"].value_counts().head(10)

In [ ]:
#customerID pas unique  
df["CustomerID"].nunique()

Analyse des distributions

In [ ]:
#distrucition de Quantity
sns.histplot(data=df, x="Quantity", bins=50)
plt.title("Distribution de Quantity")
plt.show()

In [ ]:
#distribution de unitPrice
sns.histplot(data=df, x="UnitPrice", bins=50)
plt.title("Distribution de UnitPrice")
plt.show()

Outliers

In [ ]:
sns.boxplot(data=df, x="Quantity")
plt.title("Outliers de Quantity")
plt.show()

In [ ]:
sns.boxplot(data=df, x="UnitPrice")
plt.title("Outliers de UnitPrice")
plt.show()

### Nettoyage et préparation des données

Gestion des valeurs manquantes

In [ ]:
#copy data to work in 
df_clean=df.copy()

In [ ]:
df_clean.isnull().sum()

In [ ]:
#drop rows has now coustumerID
df_clean = df_clean.dropna(subset=["CustomerID"])

Gestion des doublons

In [ ]:
# Compter les lignes complètement dupliquées (5225)
print("Nombre de doublons :", df_clean.duplicated().sum())

In [ ]:
#delete les rows qui sont identique 
df_clean = df_clean.drop_duplicates()

In [ ]:
print("Doubons dans data initiale :",df.duplicated().sum())
print("Doublons restants :", df_clean.duplicated().sum())

Gestion des valeurs invalides (Returns / cancellations)

In [ ]:
#affiche price négatif or null 
df_clean[df_clean["UnitPrice"] <=0]

In [ ]:
#affiche Quantity négatif or null 
df_clean[df_clean["Quantity"] <= 0]

In [ ]:
#affiche nombre de InvoiceNo start with C
Cancellation = df_clean[
    df_clean["InvoiceNo"].astype(str).str.startswith("C")
]

print("Nombre de Cancellation :", len(Cancellation))

In [ ]:
Cancellation.head()

In [ ]:
Cancellation["Quantity"].describe()

#Conclusion : QUantity négative have relation with Cancellation donc quantité négatif is return 

In [ ]:
# Supprimer les transactions annulées / retours
df_clean = df_clean[
    ~df_clean["InvoiceNo"].astype(str).str.startswith("C")
]

In [ ]:
print("Quantity négative restante :", (df_clean["Quantity"] < 0).sum())

In [ ]:
print("nbr UnitPrice  négative ou null  :", (df_clean["UnitPrice"] <= 0).sum())

#### Conversion des types

InvoiceDate

In [ ]:
#transform type InvoiceDate to datetime
df_clean["InvoiceDate"] = pd.to_datetime(
    df_clean["InvoiceDate"],
    errors="coerce"
)

In [ ]:
print(df_clean["InvoiceDate"].dtype)

In [ ]:
df_clean["InvoiceDate"].dtype

CustomerID

In [ ]:
#transform ID  to int 
df_clean["CustomerID"] = df_clean["CustomerID"].astype(int)

Feature Engineering (Create new Columns TotalAmount)

In [ ]:
df_clean["TotalAmount"] = (
    df_clean["Quantity"] * df_clean["UnitPrice"]
)

In [ ]:
#drop columns we dont need in our logic
df_clean = df_clean.drop(
    columns=["Description", "StockCode", "Country"]
)

In [ ]:
print(df_clean.columns)

# Vérification après nettoyage

In [ ]:
#shape
df_clean.shape

In [ ]:
#missing
df_clean.isnull().sum()

In [ ]:
#Quantity
df_clean["Quantity"].describe()

In [ ]:
#unitPrice
df_clean["UnitPrice"].describe()

In [ ]:
#date
print(df_clean["InvoiceDate"].min())
print(df_clean["InvoiceDate"].max())

In [ ]:
#nmbr Customer
df_clean["CustomerID"].nunique()

# RFM 

In [ ]:
df_clean["InvoiceDate"].max()

In [ ]:
reference_date = df_clean["InvoiceDate"].max()

Calcul de Recency

In [ ]:
rfm = df_clean.groupby("CustomerID").agg(
    LastPurchase=("InvoiceDate", "max"),
)

In [ ]:
# Calcul de Recency
rfm["Recency"] = (
    reference_date - rfm["LastPurchase"]
).dt.days

# Supprimer LastPurchase
rfm = rfm.drop(columns=["LastPurchase"])

rfm.head()

#Recency Petit = Client Achat récemment 

Calcul de Frequency (nombre de transaction / chaque client)

In [ ]:
frequency = df_clean.groupby("CustomerID")["InvoiceNo"].nunique()

In [ ]:
rfm["Frequency"] = frequency

In [ ]:
rfm.head(10)

In [ ]:
print(df_clean.shape)

Calcul de Monetary

In [ ]:
df_clean["TotalAmount"] = (
    df_clean["Quantity"] * df_clean["UnitPrice"]
)

In [ ]:
monetary = df_clean.groupby("CustomerID")["TotalAmount"].sum()

In [ ]:
rfm["Monetary"] = monetary

In [ ]:
rfm.head(10)

In [ ]:
print(rfm.columns)

Analyse distributions des variables RFM 

In [ ]:
#vérifier statistique
rfm[["Recency", "Frequency", "Monetary"]].describe()

##### Visualiser les distributions

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

liste = ["Recency", "Frequency", "Monetary"]

for variable in liste:
    sns.histplot(rfm[variable], kde=True)
    plt.title(f"Distribution de {variable}")
    plt.show()

##### Calcul moyenne et médiane 

In [ ]:
rfm[["Recency", "Frequency", "Monetary"]].agg(
    ["mean", "median", "std", "min", "max"]
)

In [ ]:
#Vérfier skewness 
rfm[["Recency", "Frequency", "Monetary"]].skew()

#Frequency and Monetory has clients with values very big 

In [ ]:
#visualiser les outliers 
fig, axes = plt.subplots(3, 1, figsize=(10, 10))

sns.boxplot(x=rfm["Recency"], ax=axes[0])
axes[0].set_title("Boxplot de Recency")

sns.boxplot(x=rfm["Frequency"], ax=axes[1])
axes[1].set_title("Boxplot de Frequency")

sns.boxplot(x=rfm["Monetary"], ax=axes[2])
axes[2].set_title("Boxplot de Monetary")

plt.tight_layout()
plt.show()

#### transformation logarithmique 

In [ ]:
rfm_log = rfm[["Recency", "Frequency", "Monetary"]].copy()

In [ ]:
rfm.skew()

In [ ]:
rfm_log = np.log1p(rfm_log)

In [ ]:
rfm_log.head()

In [ ]:
rfm_log.skew()

#Recency: asymétrie à gauche 
#Frequency :Forte asymétrie à droite
#Monetory :Asymétrie à droite

In [ ]:
for column in rfm_log.columns:
    sns.histplot(rfm_log[column], kde=True)
    plt.title(f"Distribution de {column} après transformation logarithmique")
    plt.show()

#### Standarisation 

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

rfm_scaled = scaler.fit_transform(rfm_log)

In [ ]:
rfm_scaled = pd.DataFrame(rfm_scaled,columns=["Recency", "Frequency", "Monetary"],index=rfm.index)
print(rfm_scaled.head())

In [ ]:
#moyenne
rfm_scaled.mean()

In [ ]:
#écart type 
rfm_scaled.std()

#### Segementation avec K-Means 

In [ ]:
from sklearn.cluster import KMeans

k_values = range(2, 10)

In [ ]:
# Calcule d'inerti
inertias = []

for k in k_values:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    kmeans.fit(rfm_scaled)
    inertias.append(kmeans.inertia_)

In [ ]:
plt.figure(figsize=(12, 5))

plt.plot(k_values, inertias, marker="o")

plt.xlabel("Nombre de clusters (K)")
plt.ylabel("Inertia")
plt.title("Méthode du coude (Elbow Method)")

plt.show()

#### Silhouette Score 

In [ ]:
from sklearn.metrics import silhouette_score

silhouette_scores = []

for k in k_values:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = kmeans.fit_predict(rfm_scaled)

    score = silhouette_score(rfm_scaled, labels)
    silhouette_scores.append(score)

In [ ]:
plt.figure(figsize=(12, 5))

plt.plot(k_values, silhouette_scores, marker="o")

plt.xlabel("Nombre de clusters (K)")
plt.ylabel("Silhouette Score")
plt.title("Silhouette Score pour différentes valeurs de K")

plt.show()

#### Choisir K=3 

In [ ]:
#entrainé K_means avec k=3
from sklearn.cluster import KMeans
kmeans=KMeans(n_clusters=3,random_state=42,n_init=10)
rfm["Cluster"]=kmeans.fit_predict(rfm_scaled)

In [ ]:
#combien de custmer dans chaque cluster 
rfm["Cluster"].value_counts().sort_index()

#### Analyse les groupes obtenus 

#### Calcule CA

In [ ]:
profile = rfm.groupby("Cluster").agg(
    Nb_Clients=("Recency", "count"),
    Recency=("Recency", "mean"),
    Frequency=("Frequency", "mean"),
    Monetary=("Monetary", "mean")
)

profile["CA_Total"] = rfm.groupby("Cluster")["Monetary"].sum()

profile["CA_%"] = profile["CA_Total"] / profile["CA_Total"].sum() * 100

profile

# Visualisation avec PCA

In [ ]:
from sklearn.decomposition import PCA

pca = PCA(n_components=3)  #analyser la variance expliquée et choisir combien de composantes conserver.
rfm_pca = pca.fit_transform(rfm_scaled) 

# Variance expliquée
variance_expliquee = pca.explained_variance_ratio_
print("Variance expliquée :", variance_expliquee)


# Variance cumulée
variance_cumulee = variance_expliquee.cumsum()
print("Variance cumulée :", variance_cumulee)


# Visualisation (Scree plot)
plt.plot(variance_cumulee, 'o-')
plt.axhline(y=0.80, color='r', linestyle='--')  # 80%
plt.title('Scree Plot - Variance cumulée')
plt.xlabel('Nombre de composantes')
plt.ylabel('Variance cumulée')
plt.show()

In [ ]:
#visualisation des clusters dans l'espace PCA 2D
pca_2d = PCA(n_components=2)

rfm_pca_2d = pca_2d.fit_transform(rfm_scaled)

pca_df = pd.DataFrame(
    data=rfm_pca_2d,
    columns=["PC1", "PC2"],
    index=rfm.index
)

pca_df["Cluster"] = rfm["Cluster"]
# print(pca_df.head())

In [ ]:
plt.figure(figsize=(10, 6))

sns.scatterplot(
    data=pca_df,
    x="PC1",
    y="PC2",
    hue="Cluster"
)

plt.title("Visualisation des clusters avec PCA")
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.show()

## Classification avec (Random Forest/Regression Logistique / Knn)

In [ ]:
#préparation de data 
x = rfm[["Recency", "Frequency", "Monetary"]] #variable RFM 
y = rfm["Cluster"] #Variable cible : cluster K-Means

In [ ]:
#Séparation train and test data 
from sklearn.model_selection import train_test_split
x_train, x_test, y_train, y_test = train_test_split(
    x,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y   #used to ensure that the proportion of each class in the target variable is maintained in both the training and testing sets.
    )

Fonction d'évaluation commune de Modele  

In [ ]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

def evaluate_model(model, model_name):
    
    # Entraînement
    model.fit(x_train, y_train)
    
    # Prédiction
    y_pred = model.predict(x_test)
    
    # Accuracy
    accuracy = accuracy_score(y_test, y_pred)
    print(f"Accuracy {model_name} : {accuracy:.4f}")
    
    # Classification Report
    print(f"\nClassification Report - {model_name}")
    print(classification_report(y_test, y_pred))
    
    # Confusion Matrix
    cm = confusion_matrix(y_test, y_pred)
    print("Matric De Confusion est :\n ",cm)
    
    sns.heatmap(
        cm,
        annot=True,
        fmt="d",
        xticklabels=["Cluster 0", "Cluster 1", "Cluster 2"],
        yticklabels=["Cluster 0", "Cluster 1", "Cluster 2"]
    )
    
    plt.xlabel("Cluster prédit")
    plt.ylabel("Cluster réel")
    plt.title(f"Matric  e de confusion - {model_name}")
    plt.show()
    
    return accuracy, y_pred, cm

In [ ]:
#Random Forest 
from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

accuracy_rf, y_pred_rf, cm_rf = evaluate_model(
    rf_model,
    "Random Forest"
)

In [ ]:
#Logistique Regression 
from sklearn.linear_model import LogisticRegression

lr_model = LogisticRegression(
    max_iter=1000,
    random_state=42
)

accuracy_lr, y_pred_lr, cm_lr = evaluate_model(
    lr_model,
    "Logistic Regression"
)

In [ ]:
#KNN 
from sklearn.neighbors import KNeighborsClassifier

knn_model = KNeighborsClassifier(n_neighbors=5)

accuracy_knn, y_pred_knn, cm_knn = evaluate_model(
    knn_model,
    "KNN"
)

### Importance des variables

In [ ]:
importance = rf_model.feature_importances_

for feature, value in zip(x.columns, importance):
    print(feature, value)

## Nouveau client

In [ ]:
new_client=pd.DataFrame({
    "Recency":[10],   #acheter il y a 10 jours 
    "Frequency":[12], #effectuer 12 commendes 
    "Monetary":[5000] #a dépensé 5000
})

In [ ]:
predicted_cluster = rf_model.predict(new_client)

print("Cluster prédit :", predicted_cluster[0])

### Sauvegarde de Modele 

In [ ]:
import joblib

joblib.dump(rf_model, "random_forest_model.pkl")

In [ ]:
rf_model_loaded = joblib.load("random_forest_model.pkl")

Vérifier Que permet de réaliser nouvelle prédiction correct 

In [ ]:
new_client = pd.DataFrame({
    "Recency": [10],
    "Frequency": [12],
    "Monetary": [5000]
})

In [ ]:
prediction = rf_model_loaded.predict(new_client)

print("Cluster prédit :", prediction[0])